## choice

In [ ]:
VERSION = "1.1.1"
PERCENTAGE = 100  # for train
WHICH = "training"  # training or test
# PERCENTAGE = 0
# WHICH = "test"

# "1.0.1": "pixels_out",
# "1.0.2": "ellips uniform + pixels_out",
# "1.0.3": "",
# hypercube + 40_000 + shear round 2 + 10 gal/patch + shear |0.06|

# Sextractor test
# 1.1.0: "4_000",
# 1.1.1: "40_000",
# hypercube + shear round 2 + 10 gal/patch + shear |0.06| + ellips gaussian + pixels_out

## libs

In [ ]:
import resource
import warnings
from time import time
import hiplot
import holoviews
from jax import numpy, random
from polars import read_csv
from pandas import DataFrame, concat
from pathlib import Path
import tensorflow_datasets
import panel
from astropy.stats import mad_std
from seaborn import pairplot
from utils import (
    cdf_hlr_wrt_mag,
    cdf_mag,
    cdf_ns_wrt_mag,
    get_inv_cdfs,
    getting_norms,
    save_jax_array_to_fits,
    search_folder,
)
from utils_plot import (
    final_case,
    heatmaps_inputs,
    histo_seed,
    interactive_dist,
    interactive_hist,
    interactive_pairplot,
    large_ellps,
    lhs_visu,
    cdf_inv_cdf,
    cdf_inv_cdf_case,
    cdf_inv_cdf_mag_case,
    plot_example,
    visu_inputs_ouputs,
    style_norma,
    before_after,
)

holoviews.extension("bokeh", "plotly")
panel.extension()

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

resource.setrlimit(resource.RLIMIT_NOFILE, (50000, 50000))

DIR_DATASETS = search_folder(name="DATASETS")
DIR_PISCO_EUCLID = DIR_DATASETS / "pisco_euclid"
DIR_VERSION = DIR_PISCO_EUCLID / VERSION
DIR_IMAGES = DIR_PISCO_EUCLID / f"{VERSION}_images"
DIR_IMAGES.mkdir(parents=True, exist_ok=True)
DIR_SEXTRACTOR = Path(f"{DIR_VERSION}_sextractor")
DIR_SEXTRACTOR.mkdir(parents=True, exist_ok=True)

## load all params

In [ ]:
if WHICH == "training":
    used_dataset = tensorflow_datasets.load(
        f"pisco_euclid:{VERSION}",
        split=f"train[:{PERCENTAGE}%]",
        data_dir=DIR_DATASETS,
        shuffle_files=False,
    )
elif WHICH == "test":
    used_dataset = tensorflow_datasets.load(
        f"pisco_euclid:{VERSION}",
        split=f"train[{PERCENTAGE}%:]",
        data_dir=DIR_DATASETS,
        shuffle_files=False,
    )

print(TOTAL_ELEMENTS := used_dataset.cardinality().numpy())

all_params_dataset = used_dataset.map(
    lambda x: {
        "avg_e": x["avg_e"],
        "avg_g": x["avg_g"],
        "image": x["image"],
        "gal_params": x["gal_params"],
        "seeds": x["seeds"],
    },
    deterministic=True,
)

all_params = list(all_params_dataset.as_numpy_iterator())
ellips = numpy.array([item["avg_e"] for item in all_params])
shears = numpy.array([item["avg_g"] for item in all_params])
images = numpy.array([item["image"] for item in all_params])
gal_params = numpy.array([item["gal_params"] for item in all_params])
seeds = numpy.array([item["seeds"] for item in all_params])

del used_dataset, all_params_dataset, all_params

## hypercube

In [ ]:
a, b, c = gal_params.shape
gal_params = gal_params.transpose(0, 2, 1).reshape(a * c, b)

hypercube_images = DataFrame(
    numpy.hstack([shears, ellips, seeds]), columns=["avg_g1", "avg_g2", "avg_e1", "avg_e2", "seeds"]
)
hypercube_gal_params = DataFrame(
    gal_params,
    columns=[
        "ell_1",
        "ell_2",
        "x",
        "y",
        "offset_x",
        "offset_y",
        "mag",
        "half_light_radius",
        "n_sersic",
    ],
)

del shears, ellips, seeds, gal_params, a, b, c

## compute stats

In [ ]:
if WHICH == "training":
    DataFrame(
        [
            {
                "min_images": numpy.amin(images),
                "max_images": numpy.amax(images),
                "mean_images": numpy.mean(images),
                "std_images": mad_std(images),
                "min_avg_shears": numpy.amin(hypercube_images[["avg_g1", "avg_g2"]].to_numpy()),
                "max_avg_shears": numpy.amax(hypercube_images[["avg_g1", "avg_g2"]].to_numpy()),
                "min_avg_ellips": numpy.amin(hypercube_images[["avg_e1", "avg_e2"]].to_numpy()),
                "max_avg_ellips": numpy.amax(hypercube_images[["avg_e1", "avg_e2"]].to_numpy()),
                "min_ellips": numpy.amax(hypercube_gal_params[["ell_1", "ell_2"]].to_numpy()),
                "max_ellips": numpy.amax(hypercube_gal_params[["ell_1", "ell_2"]].to_numpy()),
            }
        ]
    ).to_csv(DIR_VERSION / "statistics.csv")

## save for sextractor

In [ ]:
if not any(DIR_SEXTRACTOR.iterdir()) and PERCENTAGE == 100:
    hypercube_images.to_csv(DIR_SEXTRACTOR / "true_images_params.csv", index=False)
    hypercube_gal_params.to_csv(DIR_SEXTRACTOR / "true_galaxies_params.csv", index=False)
    save_jax_array_to_fits(images, DIR_SEXTRACTOR)

## cdfs

In [ ]:
inv_cdf_mag, inv_cdfs_ns, inv_cdfs_hlr, cases = get_inv_cdfs(
    cdf_mag, cdf_ns_wrt_mag, cdf_hlr_wrt_mag
)

In [ ]:
cdf_inv_cdf_mag_case(inv_cdf_mag, cdf_mag, DIR_IMAGES / "proba_mag")
cdf_inv_cdf(inv_cdfs_ns, cdf_ns_wrt_mag, cases, DIR_IMAGES / "proba_ns")
cdf_inv_cdf(inv_cdfs_hlr, cdf_hlr_wrt_mag, cases, DIR_IMAGES / "proba_hlr")

In [ ]:
CASE = 21.0
DEG = 4

# fmt: off
cdf_inv_cdf_case(
    CASE, inv_cdfs_hlr[CASE], cdf_hlr_wrt_mag, DIR_IMAGES / "proba_case_hlr", deg=DEG
    # CASE, inv_cdfs_ns[CASE], cdf_ns_wrt_mag, DIR_IMAGES / "proba_case_ns", deg=DEG
)
del DEG, CASE

In [ ]:
final_case(
    hypercube_images.to_numpy()[:, :-1], hypercube_images.columns, DIR_IMAGES / "hypercube_images"
)

In [ ]:
final_case(
    hypercube_gal_params.to_numpy(),
    hypercube_gal_params.columns,
    DIR_IMAGES / "hypercube_galaxy_parameters",
)

## params

In [ ]:
bins = 1000
histo_seed(hypercube_images["seeds"].to_numpy(), bins, DIR_IMAGES)
del bins

In [ ]:
bins = 13
alpha = 0.8

lhs_visu(
    hypercube_images["avg_g1"],
    hypercube_images["avg_g2"],
    alpha=alpha,
    bins=bins,
    directory=DIR_IMAGES,
)
del bins, alpha

In [ ]:
bins = 20
alpha = 0.2

lhs_visu(
    hypercube_images["avg_e1"],
    hypercube_images["avg_e2"],
    alpha=alpha,
    bins=bins,
    directory=DIR_IMAGES,
)
del bins, alpha

In [ ]:
bins = 20
alpha = 0.1

lhs_visu(hypercube_gal_params["x"], hypercube_gal_params["y"], alpha, bins, DIR_IMAGES)
del bins, alpha

In [ ]:
bins = 20
alpha = 0.1

lhs_visu(hypercube_gal_params["ell_1"], hypercube_gal_params["ell_2"], alpha, bins, DIR_IMAGES)
del bins, alpha

## pairplots inputs

In [ ]:
size = 0.75
panel.Tabs(
    (
        "PairPlot",
        interactive_pairplot(
            hypercube_images.iloc[:, :-1], size, "pairplot_plotly_hypercube", DIR_IMAGES
        ),
    )
)
del size

In [ ]:
size = 0.5
panel.Tabs(
    (
        "PairPlot",
        interactive_pairplot(
            hypercube_gal_params, size, "pairplot_plotly_galaxy_parameters", DIR_IMAGES
        ),
    )
)
del size

In [ ]:
size = 0.5
fig = pairplot(hypercube_images.iloc[:, :-1], corner=True, plot_kws={"s": size})
fig.savefig(DIR_IMAGES / "pairplot_seaborn_hypercube.pdf", dpi=300)
del size, fig

In [ ]:
size = 0.25
fig = pairplot(hypercube_gal_params, corner=True, plot_kws={"s": size})
fig.savefig(DIR_IMAGES / "pairplot_seaborn_galaxy_parameters.pdf", dpi=300)
del size, fig

## examples

In [ ]:
print(selected_number := random.permutation(random.PRNGKey(int(time())), images.shape[0])[:3])
selected_images = images[selected_number]

plot_example(selected_images, DIR_IMAGES)
del selected_number, selected_images

In [ ]:
panel.Tabs(
    (
        "Dataset Exploration",
        visu_inputs_ouputs(
            dbb={
                "inputs": hypercube_images,
                "simu": images,
                "shear": hypercube_images[["avg_g1", "avg_g2"]],
            }
        ),
    )
)

## large avg ellipticities

In [ ]:
THRESH = 0.275
conditions = abs(hypercube_images["avg_e1"]) > THRESH
# conditions = abs(hypercube_images["avg_e2"]) > THRESH

filtered_inputs = hypercube_images[conditions]
print(index_numbers := filtered_inputs.index.tolist())
print(
    select_numbers := random.permutation(random.PRNGKey(int(time())), numpy.array(index_numbers))[
        :3
    ]
)
select_images = images[select_numbers]
select_avg_e1 = hypercube_images["avg_e1"][select_numbers]
select_avg_e2 = hypercube_images["avg_e2"][select_numbers]

large_ellps(
    select_images,
    select_avg_e1,
    select_avg_e2,
    select_numbers,
    DIR_IMAGES,
    name="avg",
)

del (
    select_numbers,
    select_images,
    select_avg_e1,
    select_avg_e2,
    filtered_inputs,
    conditions,
    index_numbers,
    THRESH,
)

## large single ellipticities

In [ ]:
THRESH_1 = 0.68
THRESH_2 = 0.01
conditions = abs(hypercube_gal_params["ell_1"]) > THRESH_1
# conditions = (abs(hypercube_gal_params["ell_1"]) > THRESH_1) & (
#     abs(hypercube_gal_params["ell_2"]) < THRESH_2
# )

filtered_inputs = hypercube_gal_params[conditions]
print(index_numbers := filtered_inputs.index.tolist())
print(len(index_numbers))
print(
    select_numbers_ellps := random.permutation(
        random.PRNGKey(int(time())), numpy.array(index_numbers)
    )[:3]
)

select_single_ellps1 = hypercube_gal_params["ell_1"][select_numbers_ellps]
select_single_ellps2 = hypercube_gal_params["ell_2"][select_numbers_ellps]
select_numbers_images = select_numbers_ellps // 10  # NB_GALAXY_PER_IMAGE
select_images = images[select_numbers_images]

large_ellps(
    select_images,
    select_single_ellps1,
    select_single_ellps2,
    select_numbers_images,
    DIR_IMAGES,
    "single",
)

del (
    select_numbers_images,
    select_images,
    select_numbers_ellps,
    select_single_ellps1,
    select_single_ellps2,
    THRESH_1,
    THRESH_2,
    filtered_inputs,
    conditions,
    index_numbers,
)

## norms outputs

In [ ]:
images_norms = getting_norms(data=images)
panel.Tabs(
    ("HISTO", interactive_hist(data=images_norms)), ("DIST", interactive_dist(data=images_norms))
)

## parallel plots

In [ ]:
concatenated_inputs = concat((images_norms, hypercube_images), axis=1)
_ = hiplot.Experiment.from_dataframe(dataframe=concatenated_inputs).display(force_full_width=True)

## heatmaps

In [ ]:
heatmaps_inputs(hypercube_gal_params, concatenated_inputs, DIR_IMAGES)

## scaling

In [ ]:
param_database = read_csv(DIR_VERSION / "statistics.csv", has_header=True)
sigma = param_database["std_images"][0]

style_norma(images.flatten(), style="regular", directory=DIR_IMAGES)

In [ ]:
style_norma(
    numpy.arcsinh(images.flatten() / (sigma * 3)), style="arcsinh-sig-3", directory=DIR_IMAGES
)

In [ ]:
NUM = 0

before_after(images[NUM, :, :], DIR_IMAGES)
del NUM